# Modelling

Train and evaluate models using outputs from preprocessing.ipynb.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
)
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

pipeline_out = Path('outputs/pipeline')
X_train = pd.read_csv(pipeline_out / 'X_train.csv')
X_test = pd.read_csv(pipeline_out / 'X_test.csv')
y_train = pd.read_csv(pipeline_out / 'y_train.csv')['Attrition']
y_test = pd.read_csv(pipeline_out / 'y_test.csv')['Attrition']

X = pd.concat([X_train, X_test], axis=0, ignore_index=True)

# Scale only for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    'Logistic Regression': LogisticRegression(max_iter=200, C=0.1),
    'Random Forest': RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_split=5,
        random_state=42,
    ),
    'XGBoost': XGBClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='logloss',
        random_state=42,
    ),
}

results = []

for name, model in models.items():
    if name == 'Logistic Regression':
        model.fit(X_train_scaled, y_train)
        preds = model.predict(X_test_scaled)
        probs = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        probs = model.predict_proba(X_test)[:, 1]

    results.append({
        'model': name,
        'accuracy': round(accuracy_score(y_test, preds), 4),
        'precision': round(precision_score(y_test, preds, zero_division=0), 4),
        'recall': round(recall_score(y_test, preds, zero_division=0), 4),
        'f1_score': round(f1_score(y_test, preds, zero_division=0), 4),
        'roc_auc': round(roc_auc_score(y_test, probs), 4),
    })

comparison_df = pd.DataFrame(results).sort_values('roc_auc', ascending=False).reset_index(drop=True)
out = Path('outputs')
out.mkdir(parents=True, exist_ok=True)
comparison_df.to_csv(out / 'model_comparison.csv', index=False)
comparison_df


In [ ]:
out = Path('outputs/figures')
out.mkdir(parents=True, exist_ok=True)

# 1. Feature importance
rf = models['Random Forest']
feat_imp = pd.Series(rf.feature_importances_, index=X.columns)
feat_imp.nlargest(10).sort_values().plot(kind='barh', title='Top 10 features driving attrition')
plt.tight_layout()
plt.savefig(out / 'feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

# 2. Confusion matrix for final model
xgb = models['XGBoost']
ConfusionMatrixDisplay.from_estimator(xgb, X_test, y_test, cmap='Blues')
plt.tight_layout()
plt.savefig(out / 'confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

# 3. ROC curve for final model
RocCurveDisplay.from_estimator(xgb, X_test, y_test)
plt.tight_layout()
plt.savefig(out / 'roc_curve.png', dpi=300, bbox_inches='tight')
plt.show()
